# Seasonal Agriculture Performance Analysis

This notebook analyzes seasonal agricultural performance using the provided dataset to compare yield, profitability, resource use, and climate conditions across Kharif, Rabi, and Zaid seasons.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv("data/seasonal_agriculture_performance_dataset.csv")
df.head()


## 1. Dataset overview

The dataset contains agricultural farm records across states, districts, crops, and seasons. It includes crop, production, revenue, profit, environmental, soil, and irrigation variables.


In [ ]:
df.info()
df.shape
df.isna().sum().sort_values(ascending=False).head(10)


In [ ]:
df.duplicated().sum()
df["Season"].value_counts()


## 2. Data cleaning and preprocessing

The dataset contains a small number of missing values in numerical fields. These are filled with the median value for each column so they do not distort the analysis.


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
df["Season"] = pd.Categorical(df["Season"], categories=["Kharif", "Rabi", "Zaid"], ordered=True)
df.isna().sum().sum()


In [ ]:
season_summary = df.groupby("Season", observed=True).agg({
    "Yield_Tonnes_Ha": "mean",
    "Profit_INR": "mean",
    "Rainfall_mm": "mean",
    "Avg_Temperature_C": "mean",
    "Total_Cost_INR": "mean"
}).reset_index()
season_summary.round(2)


## 3. Seasonal comparison of productivity and profitability

Kharif generally delivers the strongest performance due to favorable rainfall and temperature conditions. Rabi remains stable, while Zaid shows lower productivity and negative average profit.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.barplot(data=season_summary, x="Season", y="Yield_Tonnes_Ha", palette="Set2", ax=axes[0,0])
axes[0,0].set_title("Average Yield by Season")
axes[0,0].set_ylabel("Yield (tonnes/ha)")

sns.barplot(data=season_summary, x="Season", y="Profit_INR", palette="Set2", ax=axes[0,1])
axes[0,1].set_title("Average Profit by Season")
axes[0,1].set_ylabel("Profit (INR)")

sns.barplot(data=season_summary, x="Season", y="Rainfall_mm", palette="Set2", ax=axes[1,0])
axes[1,0].set_title("Average Rainfall by Season")
axes[1,0].set_ylabel("Rainfall (mm)")

sns.barplot(data=season_summary, x="Season", y="Total_Cost_INR", palette="Set2", ax=axes[1,1])
axes[1,1].set_title("Average Production Cost by Season")
axes[1,1].set_ylabel("Cost (INR)")

plt.tight_layout()


In [ ]:
sns.boxplot(data=df, x="Season", y="Yield_Tonnes_Ha", palette="Set2")
plt.title("Yield Distribution Across Seasons")
plt.ylabel("Yield (tonnes/ha)")
plt.show()


In [ ]:
sns.scatterplot(data=df, x="Rainfall_mm", y="Yield_Tonnes_Ha", hue="Season", s=80, alpha=0.75)
plt.title("Rainfall vs Yield")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Yield (tonnes/ha)")
plt.legend(title="Season")
plt.show()


In [ ]:
corr_cols = ["Rainfall_mm", "Avg_Temperature_C", "Soil_Moisture_pct", "Fertilizer_kg_ha", "Yield_Tonnes_Ha", "Profit_INR"]
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap for Key Agricultural Variables")
plt.show()


## 4. Crop and irrigation insights

Profitability varies strongly across crop types and irrigation methods. Sugarcane and chilli are among the highest-profit crops, while drip irrigation supports the strongest average returns.


In [ ]:
crop_summary = df.groupby("Crop", observed=True).agg({
    "Profit_INR": "mean",
    "Yield_Tonnes_Ha": "mean"
}).reset_index().sort_values("Profit_INR", ascending=False)
crop_summary.head(10)


In [ ]:
irrigation_summary = df.groupby("Irrigation_Method", observed=True)["Profit_INR"].mean().reset_index().sort_values("Profit_INR", ascending=False)
sns.barplot(data=irrigation_summary, x="Irrigation_Method", y="Profit_INR", palette="viridis")
plt.title("Average Profit by Irrigation Method")
plt.ylabel("Profit (INR)")
plt.xticks(rotation=20)
plt.show()


## 5. Key findings and recommendations

The analysis shows that Kharif is the most productive and profitable season, while Zaid records lower output and negative average profit. Seasonal rainfall and crop suitability strongly influence performance, and drip irrigation appears to improve economic returns.

### Recommendations
- Prioritize Kharif crop planning for high-yield and income-oriented crops.
- Improve water management and irrigation efficiency during low-rainfall seasons.
- Promote high-profit crops such as sugarcane and chilli in suitable regions.
- Strengthen pest and disease monitoring during Zaid conditions.
- Use data-driven seasonal planning to minimize cost and improve profitability.


In [ ]:
best_season = season_summary.sort_values("Profit_INR", ascending=False).iloc[0]
best_crop = crop_summary.head(1).iloc[0]
best_irrigation = irrigation_summary.head(1).iloc[0]
print("Best performing season by average profit:", best_season["Season"], f"({best_season["Profit_INR"]:.2f} INR)")
print("Highest profit crop:", best_crop["Crop"], f"({best_crop["Profit_INR"]:.2f} INR)")
print("Best irrigation method:", best_irrigation["Irrigation_Method"], f"({best_irrigation["Profit_INR"]:.2f} INR)")


## 6. Conclusion

Seasonal agricultural performance varies significantly due to rainfall distribution, temperature conditions, crop type, and irrigation efficiency. By understanding these seasonal patterns, farmers and planners can improve productivity, reduce risk, and increase profitability across different seasons.
